# 02 — Silver Data Cleaning | Giai đoạn A: Tiền xử lý và xuất phiếu chấm

**HybriNews · PySpark DataFrame · Windows Local (16 logical cores)**

> **DỪNG TRƯỚC SILVER:** Notebook này **chỉ** chuẩn hóa và xử lý sơ bộ dữ liệu, tạo **hai loại phiếu chấm thủ công**, rồi dừng. **KHÔNG ghi `data/silver/`, KHÔNG tự chọn target, KHÔNG tự xóa near-duplicate.** Sau khi bạn chấm hết và gửi lại file CSV, chúng ta mới xây dựng giai đoạn B để áp dụng quyết định và xuất Silver.

**Ba dataset chính thức:** `cnn_dailymail`, `xsum`, `vietnews`. XL-Sum VI vẫn giữ nguyên trong Bronze và báo cáo EDA nhưng không tham gia bước này.

### Quy trình chính xác

1. Đọc Bronze và chuẩn hóa có kiểm soát (Unicode NFC, khoảng trắng, bảo toàn dấu `_` VietNews).
2. Source/target rỗng → loại khỏi tập tạm.
3. Source giống, target giống trong một split → giữ một bản có thứ tự ổn định.
4. Source giống, khác split → chỉ giữ split ưu tiên **test > validation > train**.
5. Source giống, target khác → xuất **toàn bộ phương án** để bạn chấm `KEEP/DROP`.
6. Source gần giống → xuất **các cặp ứng viên** để bạn chấm `DUPLICATE/NOT_DUPLICATE`.
7. Kết thúc ở **điểm dừng thủ công**. Chưa tạo dữ liệu Silver chính thức.

**Lưu ý:** bước near-duplicate sử dụng blocking + Jaccard 3-gram để tìm **ứng viên**, **không bảo đảm phát hiện tất cả** các cặp gần giống.


**Near-Duplicate Optimized:** Bước 5 dùng Spark để sinh ứng viên và Python để chấm điểm trên số lượng cặp nhỏ; không tạo Spark join hai lần trên toàn văn.

**Windows Safe Mode:** phiên bản này mặc định `local[2]`, giảm cache đồng thời và kiểm tra Python UDF trước khi chạy corpus lớn. Vẫn chỉ xuất CSV đánh giá; không ghi Silver. Nếu Spark Java bị ngắt, hãy **Restart Kernel** trước khi chạy lại.

## 0. Cấu hình dự án và đường dẫn

In [2]:
from pathlib import Path
import os, sys, csv, json, hashlib, shutil, platform
from datetime import datetime, timezone

HERE = Path.cwd().resolve()
ROOT = HERE.parent if HERE.name.lower() == "notebooks" else HERE
if not (ROOT / "notebooks").is_dir():
    raise RuntimeError(f"Không tìm thấy thư mục notebooks trong PROJECT ROOT: {ROOT}. Hãy mở notebook ở thư mục CK_BIG/notebooks.")
RAW = ROOT / "data" / "raw"
SILVER = ROOT / "data" / "silver"
REPORTS = ROOT / "reports" / "silver"
REVIEWS = REPORTS / "reviews"
for p in (REPORTS, REVIEWS):
    p.mkdir(parents=True, exist_ok=True)

# Review CSV không bao giờ được tự động ghi đè; muốn tạo lại, hãy đổi tên file cũ trước.
# Chỉ tạo reports/ để lưu kết quả chấm; KHÔNG tạo data/silver.
NEAR_MIN_CHARS = 200
NEAR_BLOCK_CHARS = 100
NEAR_MAX_BLOCK_GROUP = 30
NEAR_MIN_LENGTH_RATIO = 0.75
NEAR_JACCARD_THRESHOLD = 0.82
NEAR_MAX_CANDIDATE_PAIRS = 150000
NEAR_MAX_REVIEW_PAIRS = 5000

print('PROJECT:', ROOT)
print('PYTHON:', sys.executable)
print('CPU logical:', os.cpu_count())
print('BRONZE:', RAW)
print('REVIEW CSV:', REVIEWS)
print('SILVER OUTPUT DISABLED')


PROJECT: D:\DuLieuCuaHuu\HK1_20262027\BigData\CK_BIG
PYTHON: c:\Users\Lenovo\AppData\Local\Programs\Python\Python313\python.exe
CPU logical: 16
BRONZE: D:\DuLieuCuaHuu\HK1_20262027\BigData\CK_BIG\data\raw
REVIEW CSV: D:\DuLieuCuaHuu\HK1_20262027\BigData\CK_BIG\reports\silver\reviews
SILVER OUTPUT DISABLED


## 0.1. Cấu hình ba dataset, manifest và các hàm lưu báo cáo

In [3]:
SPLITS = ("train", "validation", "test")
SPLIT_PRIORITY = {"test": 0, "validation": 1, "train": 2}
DATASETS = {
    "cnn_dailymail": {
        "source": "article", "target": "highlights", "id": "id", "language": "en",
        "expected": {"train": 287113, "validation": 13368, "test": 11490}
    },
    "xsum": {
        "source": "document", "target": "summary", "id": "id", "language": "en",
        "expected": {"train": 204045, "validation": 11332, "test": 11334}
    },
    "vietnews": {
        "source": "article", "target": "abstract", "id": "guid", "language": "vi",
        "expected": {"train": 99134, "validation": 22184, "test": 22498}
    }
}

def write_json(path, data):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(data, ensure_ascii=False, indent=2), encoding="utf-8")

def file_sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for b in iter(lambda: f.read(8*1024*1024), b''):
            h.update(b)
    return h.hexdigest()

def write_csv(path, rows, fields):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8-sig', newline='') as f:
        w = csv.DictWriter(f, fieldnames=fields, extrasaction='ignore')
        w.writeheader()
        w.writerows(rows)

def manifest_for(name):
    f = RAW / name / 'source_manifest.json'
    if not f.exists():
        raise FileNotFoundError(f'Thiếu {f}. Chạy notebook 00 trước.')
    m = json.loads(f.read_text(encoding='utf-8'))
    if m.get('dataset') != name or not m.get('valid'):
        raise ValueError(f'Manifest sai hoặc chưa PASS: {f}')
    return m

def parquet_files(name, split):
    m = manifest_for(name)
    meta = m.get('splits', {}).get(split)
    if not meta or not meta.get('valid'):
        raise ValueError(f'{name}/{split}: manifest chưa PASS')
    paths = [RAW / name / item['path'] for item in meta['files']]
    if not paths or any(not p.is_file() for p in paths):
        raise FileNotFoundError(f'{name}/{split}: thiếu Parquet từ notebook 00')
    return [str(p) for p in paths]

## 0.2. Khởi tạo Spark trên Windows

In [5]:
# BẮT BUỘC Restart Kernel trước khi chạy sau lỗi WinError 10054/10061.
# SAFE MODE: local[2], 8 shuffle partitions, 32MiB/input task để giảm tải JVM/Python workers.
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'
os.environ['SPARK_LOCAL_HOSTNAME'] = 'localhost'

from pyspark.sql import SparkSession, functions as F, Window
from pyspark.storagelevel import StorageLevel
from pyspark.sql import types as T
import unicodedata

SPARK_CORES = min(2, os.cpu_count() or 2)  # Windows SAFE MODE: tối đa 2 tác vụ đồng thời
SHUFFLE_PARTITIONS = 8  # giảm áp lực shuffle và Python workers
spark = (SparkSession.builder
    .appName('HybriNews-02-Silver')
    .master(f'local[{SPARK_CORES}]')
    .config('spark.driver.host', '127.0.0.1')
    .config('spark.driver.bindAddress', '127.0.0.1')
    .config('spark.python.worker.reuse', 'false')
    .config('spark.sql.shuffle.partitions', str(SHUFFLE_PARTITIONS))
    .config('spark.sql.files.maxPartitionBytes', '33554432')  # 32 MiB/input task
    .config('spark.sql.adaptive.enabled', 'true')
    .config('spark.driver.memory', '4g')
    .getOrCreate())
spark.sparkContext.setLogLevel('WARN')
assert spark.sparkContext.getConf().get('spark.python.worker.reuse') == 'false', 'Restart Kernel rồi chạy lại cell Spark.'
print('Spark:', spark.version, '| Master:', spark.sparkContext.master,
      '| Worker reuse:', spark.sparkContext.getConf().get('spark.python.worker.reuse'))

@F.udf(returnType=T.StringType())
def normalize_nfc(text):
    return unicodedata.normalize('NFC', text) if text is not None else None

Spark: 4.1.2 | Master: local[2] | Worker reuse: false


## 0.3. Đọc Bronze và chuẩn hóa văn bản (giữ nguyên dấu `_` của VietNews)

In [6]:
# Chỉ chuẩn hóa Unicode NFC + whitespace; KHÔNG xóa dấu _, punctuation, HTML, hay chuyển văn bản sang lowercase.
# Lowercase chỉ dùng cho khóa so sánh/hash. Bronze gốc không bị sửa.
def clean_text_expr(col):
    x = normalize_nfc(F.coalesce(col, F.lit('')))
    x = F.regexp_replace(x, r'[\u200b\u200c\u200d\ufeff]', '')  # ký tự zero-width/BOM
    x = F.regexp_replace(x, r'\s+', ' ')
    return F.trim(x)

def read_and_normalize(name):
    cfg = DATASETS[name]
    parts = []
    for split in SPLITS:
        df = spark.read.parquet(*parquet_files(name, split))
        required = [cfg['id'], cfg['source'], cfg['target']]
        missing = set(required) - set(df.columns)
        if missing:
            raise ValueError(f'{name}/{split} thiếu các cột: {sorted(missing)}')
        df = df.select(
            F.lit(name).alias('dataset'),
            F.lit(cfg['language']).alias('language'),
            F.lit(split).alias('split'),
            F.col(cfg['id']).cast('string').alias('article_id'),
            F.col(cfg['source']).cast('string').alias('source_original'),
            F.col(cfg['target']).cast('string').alias('target_original'),
        )
        parts.append(df)
    combined = parts[0].unionByName(parts[1]).unionByName(parts[2])
    df = (combined
        .withColumn('source_text', clean_text_expr(F.col('source_original')))
        .withColumn('target_text', clean_text_expr(F.col('target_original'))))
    # Nghiêm túc tránh đánh đồng dữ liệu nguồn với labels theo dataset; task sẽ gán ở Gold.
    df = (df
        .withColumn('source_hash', F.sha2(F.lower(F.col('source_text')), 256))
        .withColumn('target_hash', F.sha2(F.lower(F.col('target_text')), 256))
        .withColumn('record_id', F.sha2(F.concat_ws('¦',
            F.col('dataset'), F.col('split'), F.coalesce('article_id', F.lit('')),
            F.col('source_hash'), F.col('target_hash')), 256)))
    return df

# Biến trạng thái cho từng dataset. Cache DISK_ONLY giúp tránh recompute lineage nhiều lần.
STATE = {}
def keep_disk(df):
    return df.persist(StorageLevel.DISK_ONLY)

def get_state(name):
    if name not in STATE:
        raise RuntimeError(f'{name}: cần chạy các cell phía trên theo thứ tự')
    return STATE[name]

## 0.4. Bước 1–4: lọc rỗng, duplicate chính xác, ưu tiên split, lập phiếu khác target

In [7]:
def stage1(name):
    """Bước 1: kiểm tra Bronze, chỉ cache bản hợp lệ. Không giữ 3 cache dữ liệu dài cùng lúc."""
    if name in STATE:
        raise RuntimeError(f'{name} đã khởi tạo. Restart Kernel trước khi chạy lại toàn bộ notebook.')
    df = read_and_normalize(name)  # lazy; không persist bản raw gốc
    expected = sum(DATASETS[name]['expected'].values())
    raw = df.count()
    if raw != expected:
        raise AssertionError(f'{name}: {raw:,} dòng, mong đợi {expected:,}; kiểm tra notebook 00/manifest.')
    bad = (F.length('source_text') == 0) | (F.length('target_text') == 0)
    # Lưu chỉ những cột thực sự cần cho bước kế tiếp, bỏ bản original (đã ở Bronze).
    need = ['dataset','language','split','article_id',
            'source_text','target_text','source_hash','target_hash','record_id']
    valid = keep_disk(df.filter(~bad).select(*need))
    try:
        n_valid = valid.count()  # duy nhất 1 count buộc thực thi UDF Unicode và cache
        n_bad = raw - n_valid  # tránh count thứ 3 trên nhánh invalid có Python UDF
        assert raw == n_bad + n_valid and n_bad >= 0
    except Exception:
        try: valid.unpersist(blocking=False)
        except Exception: pass
        raise
    STATE[name] = {'valid': valid, 'counts': {
        'raw_rows': raw, 'empty_rows': n_bad, 'after_empty': n_valid}}
    print(f'[{name}] BƯỚC 1: {raw:,} → {n_valid:,}; loại rỗng {n_bad:,}')


def stage2(name):
    s = get_state(name)
    w = (Window.partitionBy('split','source_hash','target_hash')
         .orderBy(F.coalesce(F.col('article_id'), F.lit('')), F.col('record_id')))
    ranked = s['valid'].withColumn('_rank', F.row_number().over(w))
    dropped = keep_disk(ranked.filter(F.col('_rank') > 1).drop('_rank'))
    kept = keep_disk(ranked.filter(F.col('_rank') == 1).drop('_rank'))
    n_drop, n_keep = dropped.count(), kept.count()
    assert n_drop + n_keep == s['counts']['after_empty']
    s.update({'pair_unique':kept,'exact_extra':dropped})
    s['counts'].update({'exact_removed':n_drop,'after_exact':n_keep})
    print(f'[{name}] BƯỚC 2: {n_keep:,} còn lại; loại {n_drop:,} source+target trùng trong split')
    # pair_unique / exact_extra đã materialize; giải phóng bản valid để giảm cache đĩa.
    s['valid'].unpersist(blocking=True)


def stage3(name):
    s = get_state(name)
    pr = (F.when(F.col('split') == 'test',F.lit(0))
            .when(F.col('split') == 'validation',F.lit(1))
            .otherwise(F.lit(2)))
    prioritized = s['pair_unique'].withColumn('_split_priority', pr)
    best = prioritized.groupBy('source_hash').agg(F.min('_split_priority').alias('_best'))
    ranked = prioritized.join(best, 'source_hash')
    kept = keep_disk(ranked.filter(F.col('_split_priority') == F.col('_best'))
                     .drop('_split_priority','_best'))
    dropped = keep_disk(ranked.filter(F.col('_split_priority') != F.col('_best'))
                        .drop('_split_priority','_best'))
    n_keep, n_drop = kept.count(), dropped.count()
    assert n_keep + n_drop == s['counts']['after_exact']
    s.update({'split_selected':kept,'cross_split_removed':dropped})
    s['counts'].update({'cross_split_removed':n_drop,'after_split':n_keep})
    print(f'[{name}] BƯỚC 3: {n_keep:,} ở split ưu tiên; loại {n_drop:,} ở split thấp ưu tiên')


def review_path(name, kind):
    return REVIEWS / f'{name}_{kind}.csv'


def validate_existing_review(path, expected_ids, id_field, required_fields):
    if not path.exists():
        return False
    with path.open('r', encoding='utf-8-sig', newline='') as fp:
        reader = csv.DictReader(fp)
        headers = set(reader.fieldnames or [])
        if not set(required_fields).issubset(headers):
            raise ValueError(f'CSV {path} có cấu trúc cũ/không hợp lệ. '
                             'Hãy sao lưu/đổi tên file cũ, rồi chạy lại cell để tạo mẫu mới.')
        rows = list(reader)
    current_ids = [r[id_field] for r in rows]
    if len(current_ids) != len(set(current_ids)) or set(current_ids) != set(expected_ids):
        raise ValueError(f'{path}: danh sách IDs đã thay đổi hoặc trùng. '
                         'Không ghi đè file đang có; hãy đối chiếu và sao lưu trước.')
    print('Đã có phiếu chấm khớp dữ liệu (GIỮ NGUYÊN):', path)
    return True


def stage4(name):
    s = get_state(name)
    # Phát hiện mọi source có nhiều target, kể cả khác split; các hàng ở split thấp
    # vẫn hiển thị để minh bạch nhưng không được phép chọn KEEP.
    variant_keys = keep_disk(s['pair_unique'].groupBy('source_hash')
      .agg(F.countDistinct('target_hash').alias('n_targets'))
      .filter(F.col('n_targets') > 1))
    groups = variant_keys.count()
    all_rows = (s['pair_unique'].join(variant_keys,'source_hash','inner')
      .select('dataset','source_hash','n_targets','split','article_id',
              'record_id','target_hash','source_text','target_text'))
    eligible = (s['split_selected'].select('record_id').distinct()
                .withColumn('eligible_after_split',F.lit(1)))
    all_rows = all_rows.join(eligible,'record_id','left').fillna({'eligible_after_split':0})
    items = all_rows.orderBy('source_hash','split','record_id').limit(15001).collect()
    if len(items) > 15000:
        raise RuntimeError(f'{name}: trên 15.000 dòng variant; cần xuất CSV phân tán thay vì collect.')
    fields = ['dataset','source_hash','n_targets','split','eligible_after_split',
              'article_id','record_id','target_hash','source_text','target_text',
              'review_decision','reviewer','review_notes']
    rows = []
    for item in items:
        d = item.asDict()
        d.update(review_decision='',reviewer='',review_notes='')
        rows.append(d)
    path = review_path(name, 'source_target_variants')
    if not validate_existing_review(path,[r['record_id'] for r in rows],'record_id',fields):
        write_csv(path,rows,fields)
        print('ĐÃ TẠO file chấm target:',path)
    s.update({'variant_keys':variant_keys,'variant_review_rows':len(rows)})
    s['counts'].update({'variant_groups':groups,'variant_review_rows':len(rows)})
    print(f'[{name}] BƯỚC 4: {groups} nhóm / {len(rows)} bản ghi khác target chờ bạn chọn')


## 0.5. Thuật toán Near-Duplicate tối ưu (bước 5)

Phân tích của bản trước cho thấy **bước sau khi sinh cặp ứng viên** là nút thắt: Spark phải join toàn văn hai lần, tạo hàng triệu 3-gram và sắp xếp trong cùng một Spark Job khi gọi `.collect()`.

**Bản mới chia hai phần:**

- **Spark DataFrame:** Lấy các source duy nhất, tạo blocking key, bỏ qua các block quá lớn, sinh và đếm các **cặp ứng viên bằng hash**.
- **Python driver:** Chỉ nhận các cặp hash đã giới hạn, lấy **một lần** nội dung những source tham gia ứng viên qua broadcast semi-join, rồi tính Jaccard 3-gram và xuất phiếu để bạn chấm.

Đối với CNN/DailyMail theo log trước, có khoảng **12.702 cặp** ứng viên, nên xử lý điểm số sau bước blocking ở driver sẽ tránh hai phép join phân tán tốn kém trên toàn văn hơn 300.000 bài.

**Kiểm soát:** Nếu danh sách ứng viên hoặc lượng source cần đưa về driver vượt giới hạn, notebook **báo lỗi rõ ràng** chứ không tự bỏ dữ liệu. Phép so sánh dùng tối đa 450 token/source, giữ dấu `_` và Unicode NFC. Kết quả không bảo đảm phát hiện mọi near-duplicate vì còn phụ thuộc blocking và ngưỡng.

**Lưu ý:** Đây là phép tính nhỏ **sau** khi Spark đã lọc dữ liệu lớn, không thay thế phần tiền xử lý PySpark. **Chưa ghi Silver.**

In [8]:
# Bước 5: Spark DataFrame chỉ tạo ứng viên qua blocking.
# Tính Jaccard trên tập ứng viên giới hạn bằng Python ở driver để TRÁNH hai lần Spark join
# với toàn văn bài báo và array_intersect/array_union trên một DAG khổng lồ.
# CẢNH BÁO: đây là bước PHÁT HIỆN ỨNG VIÊN, không cam kết bắt được mọi near-duplicate.
import time
import re
from functools import lru_cache

# \\w trong Python nhận dạng chữ/số Unicode và dấu gạch dưới.
# Unicode NFC đã được chuẩn hóa ở bước đọc nên dấu tiếng Việt vẫn được giữ.
NEAR_TOKEN_PATTERN = re.compile(r'\w+', re.UNICODE)
NEAR_MAX_DRIVER_SOURCES = 60000  # Ngừng minh bạch nếu ứng viên vượt khả năng RAM driver.


def near_3gram_set(text, max_tokens=450):
    """3-gram theo token theo thứ tự gốc, chuẩn hóa chữ thường; giữ dấu _ của VietNews."""
    words = NEAR_TOKEN_PATTERN.findall((text or '').lower())[:max_tokens]
    if len(words) < 3:
        return frozenset()
    return frozenset(zip(words, words[1:], words[2:]))


def near_jaccard_3gram(a, b):
    """Jaccard của hai bộ word 3-gram, mỗi đầu vào tối đa 450 token."""
    aa, bb = near_3gram_set(a), near_3gram_set(b)
    if not aa or not bb:
        return 0.0
    shared = len(aa.intersection(bb))
    total = len(aa) + len(bb) - shared
    return shared / total if total else 0.0


def prepare_near_blocks(df):
    txt = F.trim(F.regexp_replace(F.lower(F.col('source_text')),
                                   r'[^\p{L}\p{M}\p{N}_]+', ' '))
    base = (df.select('source_hash', 'source_text')
        .withColumn('_text', txt)
        .withColumn('_len', F.length('_text'))
        .filter(F.col('_len') >= NEAR_MIN_CHARS))

    first = F.substring(F.col('_text'), 1, NEAR_BLOCK_CHARS)
    last = F.expr(f'right(_text,{NEAR_BLOCK_CHARS})')
    middle = F.expr(
        f'substring(_text, greatest(1, cast(length(_text)/2 as int)-{NEAR_BLOCK_CHARS//2}), {NEAR_BLOCK_CHARS})')
    keys = F.array(
        F.struct(F.lit('head').alias('_type'), F.sha2(first, 256).alias('_signature')),
        F.struct(F.lit('middle').alias('_type'), F.sha2(middle, 256).alias('_signature')),
        F.struct(F.lit('tail').alias('_type'), F.sha2(last, 256).alias('_signature'))
    )
    return (base.select('source_hash', '_len', F.explode(keys).alias('_key'))
        .select('source_hash', '_len', F.col('_key._type').alias('_type'),
                F.col('_key._signature').alias('_signature')))


def stage5(name):
    s = get_state(name)
    start = time.perf_counter()
    previous = start
    temporary_cache = []

    def progress(number, detail):
        nonlocal previous
        now = time.perf_counter()
        print(f'[{name}] Bước {number}: {detail}; '
              f'{now-previous:.1f}s bước này / {now-start:.1f}s tổng', flush=True)
        previous = now

    def materialize_disk(df):
        cached = keep_disk(df)
        temporary_cache.append(cached)
        return cached

    try:
        # Một dòng source đại diện cho mỗi source_hash; chỉ để tìm near-duplicate.
        # Không chọn target cho các nhóm đang đợi review ở bước 4.
        w = Window.partitionBy('source_hash').orderBy('record_id')
        source_pool = materialize_disk(
            s['split_selected']
             .select('source_hash', 'record_id', 'split', 'source_text')
             .withColumn('_rn', F.row_number().over(w))
             .filter(F.col('_rn') == 1).drop('_rn'))
        n_sources = source_pool.count()
        progress('5.1', f'{n_sources:,} source duy nhất')

        blocks = materialize_disk(prepare_near_blocks(source_pool))
        n_blocks = blocks.count()
        progress('5.2', f'{n_blocks:,} khóa blocking')

        # Chỉ tạo cặp trong các block nhỏ. Block quá lớn được đếm và công bố rõ.
        sizes = materialize_disk(blocks.groupBy('_type', '_signature').count())
        skipped = sizes.filter(F.col('count') > NEAR_MAX_BLOCK_GROUP).count()
        valid_groups = sizes.filter(
            (F.col('count') >= 2) & (F.col('count') <= NEAR_MAX_BLOCK_GROUP)
        ).select('_type', '_signature')

        members = (blocks.join(valid_groups, ['_type', '_signature'], 'inner')
            .groupBy('_type', '_signature')
            .agg(F.collect_list(F.struct('source_hash', '_len')).alias('_members')))
        lefts = members.select(F.posexplode('_members').alias('_i', '_left'), '_members')
        candidate_pairs = (lefts
            .select('_i', '_left', F.posexplode('_members').alias('_j', '_right'))
            .filter(F.col('_i') < F.col('_j'))
            .filter(F.least(F.col('_left._len'), F.col('_right._len')) /
                    F.greatest(F.col('_left._len'), F.col('_right._len')) >= NEAR_MIN_LENGTH_RATIO)
            .select(
                F.least(F.col('_left.source_hash'), F.col('_right.source_hash')).alias('left_source_hash'),
                F.greatest(F.col('_left.source_hash'), F.col('_right.source_hash')).alias('right_source_hash'))
            .dropDuplicates(['left_source_hash', 'right_source_hash']))

        bounded = materialize_disk(candidate_pairs.limit(NEAR_MAX_CANDIDATE_PAIRS + 1))
        n_candidates = bounded.count()
        progress('5.3', f'{n_candidates:,} cặp ứng viên; {skipped:,} block bỏ qua')
        if n_candidates > NEAR_MAX_CANDIDATE_PAIRS:
            raise RuntimeError(f'{name}: vượt {NEAR_MAX_CANDIDATE_PAIRS:,} ứng viên. '
                'Cần thay đổi blocking, không tự động cắt dữ liệu.')

        # Chỉ COLLECT mã hash của ứng viên: khoảng 12.700 x 2 mã ở CNN/DailyMail,
        # KHÔNG collect toàn bộ bài báo, KHÔNG hai join nặng trên Spark.
        pair_rows = bounded.select('left_source_hash', 'right_source_hash').collect()
        pairs = [(r['left_source_hash'], r['right_source_hash']) for r in pair_rows]
        needed_hashes = sorted({h for pair in pairs for h in pair})
        if len(needed_hashes) > NEAR_MAX_DRIVER_SOURCES:
            raise RuntimeError(f'{name}: {len(needed_hashes):,} source ứng viên vượt '
                f'giới hạn RAM {NEAR_MAX_DRIVER_SOURCES:,}; cần giảm số cặp từ blocking.')
        progress('5.4', f'Đã nhận {len(pairs):,} cặp hash; cần lấy {len(needed_hashes):,} source')

        # 1 lần quét bảng source_pool ĐÃ CACHE để lấy chỉ các bài liên quan;
        # broadcast danh sách hash nhỏ, KHÔNG join 2 lần theo từng cặp.
        source_schema = T.StructType([T.StructField('source_hash', T.StringType(), False)])
        hash_keys = spark.createDataFrame([(h,) for h in needed_hashes], source_schema)
        selected = (source_pool.join(F.broadcast(hash_keys), 'source_hash', 'left_semi')
            .select('source_hash', 'record_id', 'split', 'source_text'))

        # Thu về từ từ, không .collect() DataFrame nguồn đầy đủ.
        info = {}
        for r in selected.toLocalIterator():
            info[r['source_hash']] = (r['record_id'], r['split'], r['source_text'])
        missing = set(needed_hashes).difference(info)
        if missing:
            raise RuntimeError(f'{name}: thiếu {len(missing)} source khi truy xuất; '
                               'không thể xuất review đầy đủ.')
        progress('5.5', f'Đã lấy đúng {len(info):,} source (không lấy 308 nghìn bài vào Python)')

        # Tính similarity trên tập ứng viên nhỏ bằng Python thuần (không tạo DAG Spark
        # 7 stages/70 tasks và không chạy array_union/array_intersect ở JVM nữa).
        # Cache 3-gram có giới hạn để tránh tăng RAM theo kích thước corpus.
        @lru_cache(maxsize=256)
        def grams_for(source_hash):
            return near_3gram_set(info[source_hash][2])

        hits = []
        for index, (left_hash, right_hash) in enumerate(pairs, 1):
            a, b = grams_for(left_hash), grams_for(right_hash)
            if a and b:
                common = len(a & b)
                union = len(a) + len(b) - common
                sim = common / union if union else 0.0
            else:
                sim = 0.0
            if sim >= NEAR_JACCARD_THRESHOLD:
                left_id, left_split, left_text = info[left_hash]
                right_id, right_split, right_text = info[right_hash]
                hits.append({
                    'dataset': name,
                    'pair_id': hashlib.sha256((left_hash + '|' + right_hash).encode()).hexdigest(),
                    'left_source_hash': left_hash, 'right_source_hash': right_hash,
                    'left_record_id': left_id, 'right_record_id': right_id,
                    'left_split': left_split, 'right_split': right_split,
                    'jaccard_3gram': sim,
                    'left_source': left_text, 'right_source': right_text,
                    'review_decision': '', 'remove_source_hash': '',
                    'reviewer': '', 'review_notes': ''
                })
            if index % 2500 == 0:
                print(f'[{name}] Bước 5.6: đã chấm {index:,}/{n_candidates:,} ứng viên', flush=True)
        hits.sort(key=lambda row: (-row['jaccard_3gram'],
                                    row['left_source_hash'], row['right_source_hash']))
        progress('5.6', f'{len(hits):,} cặp có Jaccard >= {NEAR_JACCARD_THRESHOLD}')
        if len(hits) > NEAR_MAX_REVIEW_PAIRS:
            raise RuntimeError(f'{name}: {len(hits):,} cặp đạt ngưỡng, vượt '
                f'{NEAR_MAX_REVIEW_PAIRS:,} dòng review. Không tự động bỏ sót; '
                'cần thống nhất cách xử lý trước.')

        fields = ['dataset', 'pair_id', 'left_source_hash', 'right_source_hash',
            'left_record_id', 'right_record_id', 'left_split', 'right_split',
            'jaccard_3gram', 'left_source', 'right_source', 'review_decision',
            'remove_source_hash', 'reviewer', 'review_notes']
        path = review_path(name, 'near_duplicate_candidates')
        if not validate_existing_review(path, [r['pair_id'] for r in hits], 'pair_id', fields):
            write_csv(path, hits, fields)
            print('ĐÃ TẠO file chấm near-duplicate:', path)
        progress('5.7', f'Đã kiểm tra/xuất {len(hits):,} phiếu review')

        s['counts'].update({
            'near_candidate_pairs_checked': n_candidates,
            'near_review_pairs': len(hits),
            'near_blocks_skipped': skipped,
            'near_unique_sources_checked': len(needed_hashes)
        })
        s['review_near_rows'] = len(hits)
        print(f'[{name}] HOÀN THÀNH BƯỚC 5 — CHƯA GHI SILVER. '
              f'Tổng thời gian {time.perf_counter()-start:.1f}s', flush=True)
    finally:
        for df in reversed(temporary_cache):
            try: df.unpersist(blocking=False)
            except Exception: pass

    # Giải phóng cache các bước 1–4 sau khi đã xuất hai loại phiếu chấm.
    for key in ('valid', 'pair_unique', 'exact_extra', 'split_selected',
                'cross_split_removed', 'variant_keys'):
        cached = s.get(key)
        if cached is not None:
            try: cached.unpersist(blocking=False)
            except Exception as exc: print(f'[WARN] Không giải phóng được {key}: {exc}')


### Kiểm tra sức khỏe Spark và Python worker (chạy trước CNN/DailyMail)

Cell này chạy với vài dòng thử nghiệm. Nếu bước chuẩn hóa NFC cũng làm Spark chết thì lỗi thuộc môi trường Spark/Python/Java, **không phải do riêng dataset CNN/DailyMail**. Khi đó xem phần hướng dẫn cuối notebook và kiểm tra log Java trước khi chạy dữ liệu lớn.

In [9]:
print('Test JVM / Spark:', spark.range(10).count())
smoke = spark.createDataFrame([('Ha\u0300 Nô\u0323i',), ('Hà_Nội',)], ['x'])
print('Test Python UDF / NFC:')
smoke.select(normalize_nfc(F.col('x')).alias('nfc')).show(truncate=False)
print('Spark và Python worker vượt qua bài kiểm tra nhỏ.')

Test JVM / Spark: 10
Test Python UDF / NFC:
+------+
|nfc   |
+------+
|Hà Nội|
|Hà_Nội|
+------+

Spark và Python worker vượt qua bài kiểm tra nhỏ.


---
## CNN/DailyMail — xử lý riêng từng bước

Các cell trong phần này chỉ chuẩn bị phiếu kiểm tra, **không ghi Silver**.

### CNN/DailyMail — Bước 1: Lọc source/target rỗng

In [10]:
stage1("cnn_dailymail")

[cnn_dailymail] BƯỚC 1: 311,971 → 311,971; loại rỗng 0


### CNN/DailyMail — Bước 2: Loại bản ghi trùng cả source và target trong một split

In [11]:
stage2("cnn_dailymail")

[cnn_dailymail] BƯỚC 2: 308,868 còn lại; loại 3,103 source+target trùng trong split


### CNN/DailyMail — Bước 3: Giữ split ưu tiên test > validation > train

In [10]:
stage3("cnn_dailymail")

[cnn_dailymail] BƯỚC 3: 308,867 ở split ưu tiên; loại 1 ở split thấp ưu tiên


### CNN/DailyMail — Bước 4: Xuất toàn bộ source giống, khác target để bạn chọn

In [11]:
stage4("cnn_dailymail")

Đã có phiếu chấm khớp dữ liệu (GIỮ NGUYÊN): D:\DuLieuCuaHuu\HK1_20262027\BigData\CK_BIG\reports\silver\reviews\cnn_dailymail_source_target_variants.csv
[cnn_dailymail] BƯỚC 4: 9 nhóm / 18 bản ghi khác target chờ bạn chọn


### CNN/DailyMail — Bước 5: Xuất các cặp source gần giống để bạn chấm

**Cơ chế mới:** Bước 5.1–5.3 tìm ứng viên bằng Spark; 5.4–5.7 lấy nội dung cần thiết và tính Jaccard ở driver. Nếu màn hình dừng lâu, xem log theo từng bước để xác định nguyên nhân.

In [12]:
stage5("cnn_dailymail")

[cnn_dailymail] Bước 5.1: 308,858 source duy nhất; 10.2s bước này / 10.2s tổng
[cnn_dailymail] Bước 5.2: 926,556 khóa blocking; 60.0s bước này / 70.2s tổng
[cnn_dailymail] Bước 5.3: 12,702 cặp ứng viên; 29 block bỏ qua; 5.3s bước này / 75.6s tổng
[cnn_dailymail] Bước 5.4: Đã nhận 12,702 cặp hash; cần lấy 15,437 source; 0.2s bước này / 75.8s tổng
[cnn_dailymail] Bước 5.5: Đã lấy đúng 15,437 source (không lấy 308 nghìn bài vào Python); 5.4s bước này / 81.2s tổng
[cnn_dailymail] Bước 5.6: đã chấm 2,500/12,702 ứng viên
[cnn_dailymail] Bước 5.6: đã chấm 5,000/12,702 ứng viên
[cnn_dailymail] Bước 5.6: đã chấm 7,500/12,702 ứng viên
[cnn_dailymail] Bước 5.6: đã chấm 10,000/12,702 ứng viên
[cnn_dailymail] Bước 5.6: đã chấm 12,500/12,702 ứng viên
[cnn_dailymail] Bước 5.6: 421 cặp có Jaccard >= 0.82; 6.1s bước này / 87.4s tổng
ĐÃ TẠO file chấm near-duplicate: D:\DuLieuCuaHuu\HK1_20262027\BigData\CK_BIG\reports\silver\reviews\cnn_dailymail_near_duplicate_candidates.csv
[cnn_dailymail] Bước 5.7: Đã

---
## XSum — xử lý riêng từng bước

Các cell trong phần này chỉ chuẩn bị phiếu kiểm tra, **không ghi Silver**.

### XSum — Bước 1: Lọc source/target rỗng

In [13]:
stage1("xsum")

[xsum] BƯỚC 1: 226,711 → 226,677; loại rỗng 34


### XSum — Bước 2: Loại bản ghi trùng cả source và target trong một split

In [14]:
stage2("xsum")

[xsum] BƯỚC 2: 226,574 còn lại; loại 103 source+target trùng trong split


### XSum — Bước 3: Giữ split ưu tiên test > validation > train

In [15]:
stage3("xsum")

[xsum] BƯỚC 3: 226,521 ở split ưu tiên; loại 53 ở split thấp ưu tiên


### XSum — Bước 4: Xuất toàn bộ source giống, khác target để bạn chọn

In [16]:
stage4("xsum")

ĐÃ TẠO file chấm target: D:\DuLieuCuaHuu\HK1_20262027\BigData\CK_BIG\reports\silver\reviews\xsum_source_target_variants.csv
[xsum] BƯỚC 4: 23 nhóm / 106 bản ghi khác target chờ bạn chọn


### XSum — Bước 5: Xuất các cặp source gần giống để bạn chấm

In [17]:
stage5("xsum")

[xsum] Bước 5.1: 226,485 source duy nhất; 5.0s bước này / 5.0s tổng
[xsum] Bước 5.2: 676,542 khóa blocking; 21.5s bước này / 26.4s tổng
[xsum] Bước 5.3: 8,634 cặp ứng viên; 20 block bỏ qua; 4.7s bước này / 31.1s tổng
[xsum] Bước 5.4: Đã nhận 8,634 cặp hash; cần lấy 7,475 source; 0.3s bước này / 31.4s tổng
[xsum] Bước 5.5: Đã lấy đúng 7,475 source (không lấy 308 nghìn bài vào Python); 3.2s bước này / 34.6s tổng
[xsum] Bước 5.6: đã chấm 2,500/8,634 ứng viên
[xsum] Bước 5.6: đã chấm 5,000/8,634 ứng viên
[xsum] Bước 5.6: đã chấm 7,500/8,634 ứng viên
[xsum] Bước 5.6: 141 cặp có Jaccard >= 0.82; 4.1s bước này / 38.6s tổng
ĐÃ TẠO file chấm near-duplicate: D:\DuLieuCuaHuu\HK1_20262027\BigData\CK_BIG\reports\silver\reviews\xsum_near_duplicate_candidates.csv
[xsum] Bước 5.7: Đã kiểm tra/xuất 141 phiếu review; 0.0s bước này / 38.7s tổng
[xsum] HOÀN THÀNH BƯỚC 5 — CHƯA GHI SILVER. Tổng thời gian 38.7s


---
## VietNews — xử lý riêng từng bước

Các cell trong phần này chỉ chuẩn bị phiếu kiểm tra, **không ghi Silver**.

### VietNews — Bước 1: Lọc source/target rỗng

In [18]:
stage1("vietnews")

[vietnews] BƯỚC 1: 143,816 → 143,816; loại rỗng 0


### VietNews — Bước 2: Loại bản ghi trùng cả source và target trong một split

In [19]:
stage2("vietnews")

[vietnews] BƯỚC 2: 143,816 còn lại; loại 0 source+target trùng trong split


### VietNews — Bước 3: Giữ split ưu tiên test > validation > train

In [20]:
stage3("vietnews")

[vietnews] BƯỚC 3: 143,815 ở split ưu tiên; loại 1 ở split thấp ưu tiên


### VietNews — Bước 4: Xuất toàn bộ source giống, khác target để bạn chọn

In [21]:
stage4("vietnews")

ĐÃ TẠO file chấm target: D:\DuLieuCuaHuu\HK1_20262027\BigData\CK_BIG\reports\silver\reviews\vietnews_source_target_variants.csv
[vietnews] BƯỚC 4: 0 nhóm / 0 bản ghi khác target chờ bạn chọn


### VietNews — Bước 5: Xuất các cặp source gần giống để bạn chấm

In [22]:
stage5("vietnews")

[vietnews] Bước 5.1: 143,815 source duy nhất; 4.3s bước này / 4.3s tổng
[vietnews] Bước 5.2: 431,430 khóa blocking; 21.0s bước này / 25.3s tổng
[vietnews] Bước 5.3: 1,505 cặp ứng viên; 0 block bỏ qua; 3.4s bước này / 28.8s tổng
[vietnews] Bước 5.4: Đã nhận 1,505 cặp hash; cần lấy 1,511 source; 0.2s bước này / 29.0s tổng
[vietnews] Bước 5.5: Đã lấy đúng 1,511 source (không lấy 308 nghìn bài vào Python); 2.8s bước này / 31.8s tổng
[vietnews] Bước 5.6: 231 cặp có Jaccard >= 0.82; 0.4s bước này / 32.2s tổng
ĐÃ TẠO file chấm near-duplicate: D:\DuLieuCuaHuu\HK1_20262027\BigData\CK_BIG\reports\silver\reviews\vietnews_near_duplicate_candidates.csv
[vietnews] Bước 5.7: Đã kiểm tra/xuất 231 phiếu review; 0.0s bước này / 32.3s tổng
[vietnews] HOÀN THÀNH BƯỚC 5 — CHƯA GHI SILVER. Tổng thời gian 32.3s


---
## KẾT THÚC GIAI ĐOẠN A — Xuất thống kê để đối chiếu trước khi bạn chấm

Sau khi cell này hoàn thành, **dừng tại đây**. Không có bước ghi Silver nào trong notebook này.


In [23]:
rows=[]
for name in DATASETS:
    if name not in STATE or 'near_review_pairs' not in STATE[name]['counts']:
        raise RuntimeError(f'{name}: chưa chạy đủ Bước 1–5')
    rows.append({'dataset':name, **STATE[name]['counts']})
fields=list(dict.fromkeys(k for row in rows for k in row))
summary_path=REPORTS/'pre_review_summary.csv'
write_csv(summary_path,rows,fields)
try:
    import pandas as pd
    display(pd.DataFrame(rows))
except ImportError:
    for r in rows: print(r)
print('\nĐÃ XUẤT PHIẾU CHẤM; CHƯA GHI SILVER.')
print('Báo cáo:',summary_path)
print('Thư mục các phiếu:',REVIEWS)


,dataset,raw_rows,empty_rows,after_empty,exact_removed,after_exact,cross_split_removed,after_split,variant_groups,variant_review_rows,near_candidate_pairs_checked,near_review_pairs,near_blocks_skipped,near_unique_sources_checked
0,cnn_dailymail,311971,0,311971,3103,308868,1,308867,9,18,12702,421,29,15437
1,xsum,226711,34,226677,103,226574,53,226521,23,106,8634,141,20,7475
2,vietnews,143816,0,143816,0,143816,1,143815,0,0,1505,231,0,1511



ĐÃ XUẤT PHIẾU CHẤM; CHƯA GHI SILVER.
Báo cáo: D:\DuLieuCuaHuu\HK1_20262027\BigData\CK_BIG\reports\silver\pre_review_summary.csv
Thư mục các phiếu: D:\DuLieuCuaHuu\HK1_20262027\BigData\CK_BIG\reports\silver\reviews


## Hướng dẫn bạn chấm hai loại CSV

### A. Source giống, target khác

File: `reports/silver/reviews/<dataset>_source_target_variants.csv`

- `source_hash`: mã nhóm cần chọn **duy nhất 1 target**.
- `eligible_after_split = 1`: bản ghi thuộc split được ưu tiên, **chỉ hàng này mới được phép `KEEP`**.
- Điền `review_decision = KEEP` cho **đúng 1 dòng của mỗi nhóm source_hash**, các dòng còn lại = `DROP`.
- Điền `reviewer` (người chấm); `review_notes` giải thích khi cần.
- Nếu CSV chỉ có dòng tiêu đề: dataset đó không có nhóm source khác target, không cần chấm.

### B. Source gần giống

File: `reports/silver/reviews/<dataset>_near_duplicate_candidates.csv`

- So sánh `left_source`, `right_source`, `left_split`, `right_split` và `jaccard_3gram`.
- Mỗi dòng: `review_decision = DUPLICATE` hoặc `NOT_DUPLICATE`.
- Nếu `DUPLICATE`: chọn **một** mã `source_hash` trong hai cột `left_source_hash` / `right_source_hash` để loại, ghi vào `remove_source_hash`. Nên ưu tiên giữ bài thuộc split `test > validation > train` nếu hai split khác nhau.
- Nếu `NOT_DUPLICATE`: để trống `remove_source_hash`.
- Điền `reviewer` và ghi chú cho các cặp khó phân biệt.
- File có thể chỉ chứa dòng tiêu đề khi không phát hiện cặp ứng viên.

**Cảnh báo:** Near-duplicate dựa trên blocking và Jaccard là thuật toán phát hiện ứng viên, **không cam kết exhaustive**. Không tự động coi `NOT_DUPLICATE` là nhãn cho mọi cặp không xuất hiện trong CSV.


# GIAI ĐOẠN B — Áp dụng phiếu đã chấm, kiểm tra và tạo Silver
Đặt các file dưới `reports/silver/reviews/`:
- `<dataset>_source_target_variants.csv` và `<dataset>_near_duplicate_candidates.csv` cho cả ba dataset.
- **Riêng XSum**: dùng file **`xsum_source_target_variants_drop20_groups.csv`** (bản đã chốt `DROP` cả 20 nhóm); **không dùng file review XSum cũ**.
- **Riêng XSum**: bổ sung `xsum_boilerplate_groups_to_exclude.csv` chứa đúng 20 hash nguồn phải loại toàn nhóm.
- Để `reports/silver/pre_review_summary.csv` từ giai đoạn A ở đúng vị trí cũ.

Quy trình B: **B1 → xác minh 7 CSV và quyết định → B2 xác minh lại với Bronze → B3 dựng dữ liệu đã duyệt cho từng dataset vào thư mục tạm → B4 kiểm tra → B5 chốt Silver chính thức.**

## B1 — Cấu hình và đọc phiếu review đã hoàn thành
Cell này **chưa đụng vào Spark hoặc Silver**, chỉ đọc và kiểm tra định dạng CSV và nguồn dữ liệu được phê duyệt.

In [13]:
from collections import defaultdict, Counter
import re

B_REVIEW_DIR = REVIEWS
B_SUMMARY_PATH = REPORTS / 'pre_review_summary.csv'
B_XSUM_VARIANTS = B_REVIEW_DIR / 'xsum_source_target_variants_drop20_groups.csv'
B_XSUM_EXCLUDE = B_REVIEW_DIR / 'xsum_boilerplate_groups_to_exclude.csv'
B_STAGE_ROOT = ROOT / 'data' / ('_silver_pending_reviewed_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'))
B_AUDIT_ROOT = REPORTS / 'after_review'
B_FULL_HEX = re.compile(r'^[0-9a-f]{64}$')

# Giai đoạn B không dùng lại STATE của giai đoạn A; dữ liệu được dựng lại
# từ Bronze với cùng quy tắc để tránh lệ thuộc vào cache/JVM của lần chạy trước.
B_REVIEWS = {}
B_BUILT = {}

def b_read_csv(path, required_cols):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f'Thiếu file {path}. Hãy chép file đã chấm vào reports/silver/reviews/.')
    with path.open('r', encoding='utf-8-sig', newline='') as fh:
        reader = csv.DictReader(fh)
        headers = reader.fieldnames or []
        missing = set(required_cols) - set(headers)
        if missing:
            raise ValueError(f'{path.name}: thiếu cột {sorted(missing)}')
        rows = [dict(r) for r in reader]
    return rows

def b_hash_ok(value):
    return bool(B_FULL_HEX.fullmatch(value or ''))

def b_priority(split):
    if split not in SPLIT_PRIORITY:
        raise ValueError(f'Split không hợp lệ: {split}')
    return SPLIT_PRIORITY[split]

print('Thư mục review:', B_REVIEW_DIR)
print('Bản XSum đã loại 20 nhóm:', B_XSUM_VARIANTS)
print('Danh sách nguồn XSum phải loại:', B_XSUM_EXCLUDE)
print('Thư mục tạm:', B_STAGE_ROOT)
print('CHƯA GHI SILVER')

Thư mục review: D:\DuLieuCuaHuu\HK1_20262027\BigData\CK_BIG\reports\silver\reviews
Bản XSum đã loại 20 nhóm: D:\DuLieuCuaHuu\HK1_20262027\BigData\CK_BIG\reports\silver\reviews\xsum_source_target_variants_drop20_groups.csv
Danh sách nguồn XSum phải loại: D:\DuLieuCuaHuu\HK1_20262027\BigData\CK_BIG\reports\silver\reviews\xsum_boilerplate_groups_to_exclude.csv
Thư mục tạm: D:\DuLieuCuaHuu\HK1_20262027\BigData\CK_BIG\data\_silver_pending_reviewed_20261110T062316Z
CHƯA GHI SILVER


## B2 — Kiểm tra toàn bộ quyết định trước khi chạy Spark

- Mỗi nhóm source–target variants: **đúng một `KEEP`**, riêng 20 nhóm XSum trong whitelist được **`DROP` toàn bộ**.
- Mỗi cặp near-duplicate: `DUPLICATE`/`NOT_DUPLICATE` hợp lệ; hash loại phải nằm trong cặp, không được loại nhầm split ưu tiên cao.
- Mỗi cặp được đánh `DUPLICATE` phải có **ít nhất một source bị loại**; cặp `NOT_DUPLICATE` không tự động loại.
- Số nhóm và số dòng review phải khớp `pre_review_summary.csv` của giai đoạn A.

Nếu có mâu thuẫn, **dừng tại đây**, chưa tạo dữ liệu mới.

In [14]:
def b_validate_all_reviews():
    summary_rows = b_read_csv(B_SUMMARY_PATH, ['dataset','raw_rows','empty_rows',
        'after_empty','exact_removed','after_exact','cross_split_removed',
        'after_split','variant_groups','variant_review_rows','near_review_pairs','near_blocks_skipped'])
    summary = {r['dataset']: r for r in summary_rows}
    if set(summary) != set(DATASETS):
        raise ValueError(f'pre_review_summary.csv không khớp 3 dataset: {sorted(summary)}')

    boiler_rows = b_read_csv(B_XSUM_EXCLUDE, ['dataset','source_hash','decision'])
    boiler = {r['source_hash'] for r in boiler_rows}
    if len(boiler_rows) != 20 or len(boiler) != 20 or any(
        r['dataset'] != 'xsum' or r['decision'] != 'DROP_ALL' or not b_hash_ok(r['source_hash'])
        for r in boiler_rows):
        raise ValueError('Danh sách XSum boilerplate phải có đúng 20 hash và decision=DROP_ALL')

    checked = {}
    for name in DATASETS:
        vp = B_XSUM_VARIANTS if name == 'xsum' else B_REVIEW_DIR / f'{name}_source_target_variants.csv'
        np = B_REVIEW_DIR / f'{name}_near_duplicate_candidates.csv'
        var_rows = b_read_csv(vp, ['dataset','source_hash','record_id','split',
            'eligible_after_split','review_decision','target_hash'])
        near_rows = b_read_csv(np, ['dataset','pair_id','left_source_hash','right_source_hash',
            'left_split','right_split','review_decision','remove_source_hash'])
        exp = summary[name]
        if len(var_rows) != int(exp['variant_review_rows']):
            raise ValueError(f'{name}: số dòng variants khác thống kê giai đoạn A')
        if len(near_rows) != int(exp['near_review_pairs']):
            raise ValueError(f'{name}: số dòng near-duplicate khác thống kê giai đoạn A')

        variants = defaultdict(list)
        seen_ids = set()
        for r in var_rows:
            h, rid = r['source_hash'], r['record_id']
            if r['dataset'] != name or not b_hash_ok(h) or not b_hash_ok(rid):
                raise ValueError(f'{name}: dataset/hash/record_id trong variant không hợp lệ')
            if rid in seen_ids:
                raise ValueError(f'{name}: record_id variant bị lặp: {rid}')
            seen_ids.add(rid)
            if r['eligible_after_split'] not in ('0','1') or r['review_decision'] not in ('KEEP','DROP'):
                raise ValueError(f'{name}: eligible_after_split / KEEP-DROP chưa hoàn chỉnh')
            b_priority(r['split'])
            variants[h].append(r)
        if len(variants) != int(exp['variant_groups']):
            raise ValueError(f'{name}: số nhóm variant không khớp EDA Phase A')

        allowed_drop_all = boiler if name == 'xsum' else set()
        if name == 'xsum' and not allowed_drop_all.issubset(variants):
            raise ValueError('XSum: một số hash boilerplate không còn trong phiếu variants')
        keep_record = {}
        for h, rows in variants.items():
            picked = [r for r in rows if r['review_decision'] == 'KEEP']
            if h in allowed_drop_all:
                if picked:
                    raise ValueError(f'XSum boilerplate {h[:12]} vẫn có KEEP')
            else:
                if len(picked) != 1 or picked[0]['eligible_after_split'] != '1':
                    raise ValueError(f'{name} source {h[:12]} phải có đúng 1 KEEP ở split ưu tiên')
                keep_record[h] = picked[0]['record_id']
            # Đảm bảo một source có thể xuất hiện ở một số split, nhưng các dòng
            # có eligible=1 phải ở split cao nhất hiện diện trong phiếu.
            highest = min(b_priority(r['split']) for r in rows)
            for r in rows:
                if (r['eligible_after_split'] == '1') != (b_priority(r['split']) == highest):
                    raise ValueError(f'{name}: eligible_after_split sai cho {h[:12]}')

        seen_pairs = set()
        near_removed = set()
        near_edges = []
        for r in near_rows:
            a, b = r['left_source_hash'], r['right_source_hash']
            if r['dataset'] != name or not b_hash_ok(a) or not b_hash_ok(b) or a >= b:
                raise ValueError(f'{name}: cặp near-duplicate không có hash theo thứ tự chuẩn')
            expected_id = hashlib.sha256((a+'|'+b).encode('utf-8')).hexdigest()
            if r['pair_id'] != expected_id or expected_id in seen_pairs:
                raise ValueError(f'{name}: pair_id không hợp lệ hoặc bị lặp')
            seen_pairs.add(expected_id)
            choice, removed = r['review_decision'], (r['remove_source_hash'] or '').strip()
            if choice == 'DUPLICATE':
                if removed not in (a,b):
                    raise ValueError(f'{name}: DUPLICATE chưa chọn đúng hash cần loại')
                survivor = b if removed == a else a
                r_split = r['left_split'] if removed == a else r['right_split']
                s_split = r['right_split'] if removed == a else r['left_split']
                if b_priority(r_split) < b_priority(s_split):
                    raise ValueError(f'{name}: near review loại split ưu tiên cao: {removed[:12]}')
                near_removed.add(removed)
                near_edges.append((a,b))
            elif choice == 'NOT_DUPLICATE':
                if removed:
                    raise ValueError(f'{name}: NOT_DUPLICATE không được có remove_source_hash')
            else:
                raise ValueError(f'{name}: còn phiếu near chưa chấm')

        # Nếu đã chấm là DUPLICATE, sau áp dụng các hash được duyệt không được
        # giữ cả hai đầu của một cạnh. Không tự coi các cạnh NOT_DUPLICATE là trùng.
        all_removed = near_removed | allowed_drop_all
        conflicts = [(a,b) for a,b in near_edges if a not in all_removed and b not in all_removed]
        if conflicts:
            raise ValueError(f'{name}: {len(conflicts)} cặp DUPLICATE vẫn còn cả 2 source')

        checked[name] = {
            'variant_rows':var_rows,'near_rows':near_rows,'variant_groups':set(variants),
            'keep_record':keep_record,'drop_all':set(allowed_drop_all),
            'near_removed':near_removed,'near_edges':near_edges,
            'expected':{k:int(exp[k]) for k in ['raw_rows','empty_rows','after_empty',
                'exact_removed','after_exact','cross_split_removed','after_split',
                'variant_groups','variant_review_rows','near_review_pairs','near_blocks_skipped']},
            'source_files': {str(vp):file_sha256(vp),str(np):file_sha256(np)}
        }
        print(f'[{name}] HỢP LỆ: variants {len(variants)} nhóm '
              f'({len(keep_record)} giữ 1, {len(allowed_drop_all)} loại cả nhóm); '
              f'near {len(near_rows)} cặp / {len(near_removed)} source được chọn loại')

    checked['xsum']['source_files'][str(B_XSUM_EXCLUDE)] = file_sha256(B_XSUM_EXCLUDE)
    for entry in checked.values():
        entry['source_files'][str(B_SUMMARY_PATH)] = file_sha256(B_SUMMARY_PATH)
    return checked

B_REVIEWS = b_validate_all_reviews()
print('ĐÃ KIỂM TRA ĐỦ REVIEW CỦA CẢ BA DATASET — CHƯA GHI SILVER')


[cnn_dailymail] HỢP LỆ: variants 9 nhóm (9 giữ 1, 0 loại cả nhóm); near 421 cặp / 409 source được chọn loại
[xsum] HỢP LỆ: variants 23 nhóm (3 giữ 1, 20 loại cả nhóm); near 141 cặp / 52 source được chọn loại
[vietnews] HỢP LỆ: variants 0 nhóm (0 giữ 1, 0 loại cả nhóm); near 231 cặp / 30 source được chọn loại
ĐÃ KIỂM TRA ĐỦ REVIEW CỦA CẢ BA DATASET — CHƯA GHI SILVER


## B3 — Dựng lại dữ liệu sau bước 1–3, áp dụng các phiếu đã duyệt

Lưu ý quan trọng: **không chạy lại `stage5()`**; chỉ dựng lại dữ liệu từ Bronze bằng cùng normalization, exact-dedup và split-priority. Kiểm tra tất cả source trong phiếu có xuất hiện đúng ở dữ liệu đã chuẩn bị.

Mỗi dataset sẽ được kiểm tra, xuất **Silver tạm** và báo cáo loại dữ liệu. **Silver chính thức chỉ được chốt khi cả ba dataset đạt kiểm định ở B5**.

In [15]:
def b_flag_hash(col, values):
    return F.col(col).isin(*sorted(values)) if values else F.lit(False)

def b_keys_dataframe(values, col_name):
    schema = T.StructType([T.StructField(col_name, T.StringType(), False)])
    return spark.createDataFrame([(x,) for x in sorted(values)], schema)

def b_small_lookup(rows, fields):
    schema = T.StructType([T.StructField(k,T.StringType(),False) for k in fields])
    return spark.createDataFrame([tuple(x) for x in rows], schema)

def b_check_review_membership(name, exact, prepared):
    """Rà soát review IDs/hash thật từ Bronze; không tin mù quáng CSV sửa tay."""
    review = B_REVIEWS[name]
    actual_variants = {r['source_hash'] for r in
        (exact.groupBy('source_hash')
         .agg(F.countDistinct('target_hash').alias('n'))
         .filter(F.col('n')>1).select('source_hash').collect())}
    if actual_variants != review['variant_groups']:
        raise ValueError(f'{name}: nhóm source-target khác dữ liệu Bronze / Phase A; DỪNG')

    var = review['variant_rows']
    if var:
        in_review = b_small_lookup([
            (r['record_id'],r['source_hash'],r['target_hash'],r['split']) for r in var],
            ['record_id','source_hash','target_hash','split'])
        actual_rows = exact.join(F.broadcast(in_review),
            on=['record_id','source_hash','target_hash','split'],how='inner').count()
        if actual_rows != len(var):
            raise ValueError(f'{name}: {actual_rows}/{len(var)} dòng review variant khớp Bronze. DỪNG')
        eligible_ids = {r['record_id'] for r in var if r['eligible_after_split']=='1'}
        if eligible_ids:
            elig = b_keys_dataframe(eligible_ids,'record_id')
            found = prepared.join(F.broadcast(elig),on='record_id',how='inner').count()
            if found != len(eligible_ids):
                raise ValueError(f'{name}: eligible_after_split không khớp dữ liệu Stage 3')

    near_hashes = set()
    for r in review['near_rows']:
        near_hashes.update((r['left_source_hash'],r['right_source_hash']))
    if near_hashes:
        near_keys = b_keys_dataframe(near_hashes, 'source_hash')
        found = (prepared.select('source_hash').dropDuplicates()
            .join(F.broadcast(near_keys),'source_hash','inner').count())
        if found != len(near_hashes):
            raise ValueError(f'{name}: {found}/{len(near_hashes)} source gần giống khớp Bronze. DỪNG')


def b_process_dataset(name):
    if set(B_REVIEWS) != set(DATASETS):
        raise RuntimeError('Cần chạy cell B2 để kiểm tra đủ phiếu trước')
    if name in B_BUILT:
        raise RuntimeError(f'{name} đã tạo Silver tạm trong phiên làm việc này')
    review = B_REVIEWS[name]
    for path, digest in review['source_files'].items():
        if file_sha256(path) != digest:
            raise RuntimeError(f'Phiếu review đã thay đổi sau kiểm tra: {path}')
    temporary = B_STAGE_ROOT / name
    if temporary.exists():
        raise FileExistsError(f'Silver tạm đã tồn tại: {temporary}; không ghi đè')
    if (SILVER / name).exists():
        raise FileExistsError(f'Silver chính thức đã tồn tại: {SILVER/name}; không tự ghi đè')

    saved = []
    def cache(x):
        y = x.persist(StorageLevel.DISK_ONLY)
        saved.append(y)
        return y
    def audit(df, reason):
        return (df.select('dataset','language','split','article_id','record_id',
            'source_hash','target_hash')
            .withColumn('removal_reason',F.lit(reason)))
    expected = review['expected']
    try:
        raw = cache(read_and_normalize(name))
        n_raw = raw.count()
        if n_raw != expected['raw_rows']:
            raise ValueError(f'{name}: Bronze khác snapshot Phase A: {n_raw} vs {expected["raw_rows"]}')
        is_bad = (F.length('source_text') == 0) | (F.length('target_text') == 0)
        valid = cache(raw.filter(~is_bad))
        n_valid = valid.count()
        if n_raw-n_valid != expected['empty_rows'] or n_valid != expected['after_empty']:
            raise ValueError(f'{name}: số bản ghi rỗng sau chuẩn hóa thay đổi')
        invalid_audit = audit(raw.filter(is_bad),'EMPTY_SOURCE_OR_TARGET')

        w = (Window.partitionBy('split','source_hash','target_hash')
            .orderBy(F.coalesce(F.col('article_id'),F.lit('')),F.col('record_id')))
        ranked = cache(valid.withColumn('_dedup_rank', F.row_number().over(w)))
        exact = cache(ranked.filter(F.col('_dedup_rank') == 1).drop('_dedup_rank'))
        exact_audit = audit(ranked.filter(F.col('_dedup_rank') > 1), 'EXACT_SOURCE_TARGET_DUPLICATE')
        n_exact = exact.count()
        if n_valid-n_exact != expected['exact_removed'] or n_exact != expected['after_exact']:
            raise ValueError(f'{name}: exact duplicates khác Phase A')

        priority = (F.when(F.col('split')=='test',F.lit(0))
                    .when(F.col('split')=='validation',F.lit(1)).otherwise(F.lit(2)))
        prior = exact.withColumn('_priority',priority)
        first = prior.groupBy('source_hash').agg(F.min('_priority').alias('_best'))
        selected = cache(prior.join(first,'source_hash')
            .filter(F.col('_priority') == F.col('_best'))
            .drop('_priority','_best'))
        split_audit = audit(prior.join(first,'source_hash')
            .filter(F.col('_priority') != F.col('_best')),
            'CROSS_SPLIT_DUPLICATE')
        n_selected = selected.count()
        if n_exact-n_selected != expected['cross_split_removed'] or n_selected != expected['after_split']:
            raise ValueError(f'{name}: split-priority khác Phase A')

        b_check_review_membership(name, exact, selected)

        keep_rules = b_small_lookup([(h,rid) for h,rid in review['keep_record'].items()],
                                    ['source_hash','_approved_record_id'])
        checked = selected.join(F.broadcast(keep_rules),'source_hash','left')
        banned = b_flag_hash('source_hash',review['drop_all'])
        near_removed = b_flag_hash('source_hash',review['near_removed'])
        is_variant_other = (b_flag_hash('source_hash',set(review['keep_record'])) &
                            (F.col('record_id') != F.col('_approved_record_id')))
        tagged = cache(checked.withColumn('removal_reason',
            F.when(banned,F.lit('BOILERPLATE_SOURCE'))
             .when(near_removed,F.lit('NEAR_DUPLICATE_MANUAL'))
             .when(is_variant_other,F.lit('VARIANT_TARGET_NOT_SELECTED'))
             .otherwise(F.lit(None).cast('string'))).drop('_approved_record_id'))

        final_cols = ['dataset','language','split','article_id','record_id',
            'source_text','target_text','source_hash','target_hash']
        clean = cache(tagged.filter(F.col('removal_reason').isNull()).select(*final_cols))
        clean_count = clean.count()
        removed = n_selected-clean_count
        if removed < 0 or clean_count == 0:
            raise ValueError(f'{name}: số bản ghi sau review bất thường')
        if clean.groupBy('source_hash').count().filter(F.col('count')>1).limit(1).count():
            raise AssertionError(f'{name}: vẫn còn >1 target hoặc split cho cùng source')
        if clean.filter((F.length('source_text')==0)|(F.length('target_text')==0)).limit(1).count():
            raise AssertionError(f'{name}: còn source/target rỗng')
        if review['drop_all'] and clean.filter(b_flag_hash('source_hash',review['drop_all'])).limit(1).count():
            raise AssertionError(f'{name}: chưa loại hết source XSum boilerplate')
        if review['near_removed'] and clean.filter(b_flag_hash('source_hash',review['near_removed'])).limit(1).count():
            raise AssertionError(f'{name}: chưa loại hết near-duplicate theo phiếu')

        # Xuất dữ liệu tạm: chưa công bố chính thức dưới data/silver.
        temporary.parent.mkdir(parents=True,exist_ok=True)
        (clean.write.mode('errorifexists').partitionBy('split').parquet(str(temporary)))
        verify = spark.read.parquet(str(temporary))
        if verify.count() != clean_count:
            raise AssertionError(f'{name}: lỗi số dòng khi đọc lại Silver tạm')
        actual_split = {r['split']:r['count'] for r in verify.groupBy('split').count().collect()}
        if set(actual_split) != set(SPLITS):
            raise AssertionError(f'{name}: thiếu split trong Silver tạm')

        audit_cols=['dataset','language','split','article_id','record_id','source_hash',
                    'target_hash','removal_reason']
        review_audit = tagged.filter(F.col('removal_reason').isNotNull()).select(*audit_cols)
        all_audit = (invalid_audit.unionByName(exact_audit)
                    .unionByName(split_audit).unionByName(review_audit))
        if n_raw != clean_count + all_audit.count():
            raise AssertionError(f'{name}: audit trước/sau không cân bằng')
        audit_path = B_AUDIT_ROOT / name / 'exclusions'
        all_audit.write.mode('errorifexists').parquet(str(audit_path))
        reason_counts = {r['removal_reason']:r['count'] for r in
            all_audit.groupBy('removal_reason').count().collect()}
        stats = {
            'dataset':name,'raw_rows':n_raw,'empty_removed':n_raw-n_valid,
            'exact_removed':n_valid-n_exact,'cross_split_removed':n_exact-n_selected,
            'after_split':n_selected,'review_removed':removed,'silver_rows':clean_count,
            'by_split':actual_split,'removed_by_reason':reason_counts,
            'near_blocks_skipped':review['expected']['near_blocks_skipped'],
            'review_decisions_sha256':review['source_files'],
            'staging_directory':str(temporary),'exclusion_directory':str(audit_path)
        }
        B_BUILT[name]=stats
        write_json(B_AUDIT_ROOT / name / 'reviewed_cleaning_summary.json',stats)
        print(f'[{name}] {n_raw:,} đầu vào → {clean_count:,} Silver tạm; '
              f'{n_raw-clean_count:,} dòng được loại; test={actual_split.get("test",0):,}, '
              f'validation={actual_split.get("validation",0):,}, train={actual_split.get("train",0):,}')
        print('Lý do loại:',reason_counts)
        return stats
    finally:
        for x in reversed(saved):
            try: x.unpersist(blocking=False)
            except Exception: pass


### CNN/DailyMail — Áp dụng review và tạo Silver tạm

Kiểm tra quyết định 9 nhóm target và 421 cặp gần giống. **Chưa ghi Silver chính thức.**

In [16]:
b_process_dataset('cnn_dailymail')

[cnn_dailymail] 311,971 đầu vào → 308,449 Silver tạm; 3,522 dòng được loại; test=11,477, validation=13,361, train=283,611
Lý do loại: {'EXACT_SOURCE_TARGET_DUPLICATE': 3103, 'CROSS_SPLIT_DUPLICATE': 1, 'VARIANT_TARGET_NOT_SELECTED': 9, 'NEAR_DUPLICATE_MANUAL': 409}


{'dataset': 'cnn_dailymail',
 'raw_rows': 311971,
 'empty_removed': 0,
 'exact_removed': 3103,
 'cross_split_removed': 1,
 'after_split': 308867,
 'review_removed': 418,
 'silver_rows': 308449,
 'by_split': {'train': 283611, 'validation': 13361, 'test': 11477},
 'removed_by_reason': {'EXACT_SOURCE_TARGET_DUPLICATE': 3103,
  'CROSS_SPLIT_DUPLICATE': 1,
  'VARIANT_TARGET_NOT_SELECTED': 9,
  'NEAR_DUPLICATE_MANUAL': 409},
 'near_blocks_skipped': 29,
 'review_decisions_sha256': {'D:\\DuLieuCuaHuu\\HK1_20262027\\BigData\\CK_BIG\\reports\\silver\\reviews\\cnn_dailymail_source_target_variants.csv': '5210ba513ffff46777d0a5bf227edfce8242d8096c6b17c9988defd29109907b',
  'D:\\DuLieuCuaHuu\\HK1_20262027\\BigData\\CK_BIG\\reports\\silver\\reviews\\cnn_dailymail_near_duplicate_candidates.csv': '6d155b45a21e947bb1c34acbcd106ddafd89a4397e0e1c6dd10a473399f746d6',
  'D:\\DuLieuCuaHuu\\HK1_20262027\\BigData\\CK_BIG\\reports\\silver\\pre_review_summary.csv': 'cede8a2c9d0234b3f73fa4ff4c7b3fbc11dde904f537ec

### XSum — Áp dụng review và loại 20 nhóm boilerplate

Kiểm tra file `xsum_source_target_variants_drop20_groups.csv`, loại **toàn bộ 20 source_hash** trong whitelist, giữ một target của 3 nhóm còn lại, áp dụng 141 cặp gần giống. **Chưa ghi Silver chính thức.**

In [17]:
b_process_dataset('xsum')

[xsum] 226,711 đầu vào → 226,413 Silver tạm; 298 dòng được loại; test=11,326, validation=11,321, train=203,766
Lý do loại: {'EMPTY_SOURCE_OR_TARGET': 34, 'EXACT_SOURCE_TARGET_DUPLICATE': 103, 'CROSS_SPLIT_DUPLICATE': 53, 'BOILERPLATE_SOURCE': 53, 'NEAR_DUPLICATE_MANUAL': 52, 'VARIANT_TARGET_NOT_SELECTED': 3}


{'dataset': 'xsum',
 'raw_rows': 226711,
 'empty_removed': 34,
 'exact_removed': 103,
 'cross_split_removed': 53,
 'after_split': 226521,
 'review_removed': 108,
 'silver_rows': 226413,
 'by_split': {'train': 203766, 'test': 11326, 'validation': 11321},
 'removed_by_reason': {'EMPTY_SOURCE_OR_TARGET': 34,
  'EXACT_SOURCE_TARGET_DUPLICATE': 103,
  'CROSS_SPLIT_DUPLICATE': 53,
  'BOILERPLATE_SOURCE': 53,
  'NEAR_DUPLICATE_MANUAL': 52,
  'VARIANT_TARGET_NOT_SELECTED': 3},
 'near_blocks_skipped': 20,
 'review_decisions_sha256': {'D:\\DuLieuCuaHuu\\HK1_20262027\\BigData\\CK_BIG\\reports\\silver\\reviews\\xsum_source_target_variants_drop20_groups.csv': '3c695c7e17b97a038b50598933798f893568a1c08efdba8ba4cf443e53e4ec97',
  'D:\\DuLieuCuaHuu\\HK1_20262027\\BigData\\CK_BIG\\reports\\silver\\reviews\\xsum_near_duplicate_candidates.csv': '114968cdb1feecf89bb1a9ef0f2954f8744eab88718d308e0acdc4368600adb5',
  'D:\\DuLieuCuaHuu\\HK1_20262027\\BigData\\CK_BIG\\reports\\silver\\reviews\\xsum_boilerplate

### VietNews — Áp dụng review các cặp gần giống

Không có source–target variants; áp dụng 231 cặp gần giống. Giữ nguyên dấu `_` trong biểu diễn đã tách từ. **Chưa ghi Silver chính thức.**

In [18]:
b_process_dataset('vietnews')

[vietnews] 143,816 đầu vào → 143,785 Silver tạm; 31 dòng được loại; test=22,498, validation=22,182, train=99,105
Lý do loại: {'CROSS_SPLIT_DUPLICATE': 1, 'NEAR_DUPLICATE_MANUAL': 30}


{'dataset': 'vietnews',
 'raw_rows': 143816,
 'empty_removed': 0,
 'exact_removed': 0,
 'cross_split_removed': 1,
 'after_split': 143815,
 'review_removed': 30,
 'silver_rows': 143785,
 'by_split': {'train': 99105, 'test': 22498, 'validation': 22182},
 'removed_by_reason': {'CROSS_SPLIT_DUPLICATE': 1,
  'NEAR_DUPLICATE_MANUAL': 30},
 'near_blocks_skipped': 0,
 'review_decisions_sha256': {'D:\\DuLieuCuaHuu\\HK1_20262027\\BigData\\CK_BIG\\reports\\silver\\reviews\\vietnews_source_target_variants.csv': 'cc1e50a70404e9ed4101e95b26ab213f729ad2ae90c1d29dd0a1e7782fcab519',
  'D:\\DuLieuCuaHuu\\HK1_20262027\\BigData\\CK_BIG\\reports\\silver\\reviews\\vietnews_near_duplicate_candidates.csv': 'c058552a65ce963c5ea755736c0802145674f166b9b459017a178bb536ce7ded',
  'D:\\DuLieuCuaHuu\\HK1_20262027\\BigData\\CK_BIG\\reports\\silver\\pre_review_summary.csv': 'cede8a2c9d0234b3f73fa4ff4c7b3fbc11dde904f537ec884d10e9c810e44668'},
 'staging_directory': 'D:\\DuLieuCuaHuu\\HK1_20262027\\BigData\\CK_BIG\\data\

## B4 — Tổng hợp và kiểm tra cả ba bộ Silver tạm

Đây là điểm chốt an toàn: **nếu chưa xử lý xong đủ 3 dataset, không được công bố Silver**. Các số liệu được ghi vào `reports/silver/after_review/`.

In [19]:
if set(B_BUILT) != set(DATASETS):
    raise RuntimeError(f'Chưa xử lý đủ ba dataset: còn thiếu {set(DATASETS)-set(B_BUILT)}')
B_SUMMARY_ROWS = []
for name in DATASETS:
    stats = B_BUILT[name]
    B_SUMMARY_ROWS.append({k:stats[k] for k in
        ['dataset','raw_rows','empty_removed','exact_removed','cross_split_removed',
         'after_split','review_removed','silver_rows']})
    if stats['raw_rows'] != stats['silver_rows'] + sum(stats['removed_by_reason'].values()):
        raise AssertionError(f'{name}: sai cân bằng bản ghi')
write_csv(B_AUDIT_ROOT/'final_cleaning_summary.csv',B_SUMMARY_ROWS,list(B_SUMMARY_ROWS[0]))
try:
    import pandas as pd
    display(pd.DataFrame(B_SUMMARY_ROWS))
except ImportError:
    for row in B_SUMMARY_ROWS: print(row)
print('3/3 DATASET ĐÃ KIỂM TRA; SILVER CHÍNH THỨC CHƯA ĐƯỢC CHỐT')


,dataset,raw_rows,empty_removed,exact_removed,cross_split_removed,after_split,review_removed,silver_rows
0,cnn_dailymail,311971,0,3103,1,308867,418,308449
1,xsum,226711,34,103,53,226521,108,226413
2,vietnews,143816,0,0,1,143815,30,143785


3/3 DATASET ĐÃ KIỂM TRA; SILVER CHÍNH THỨC CHƯA ĐƯỢC CHỐT


## B5 — Chốt Silver chính thức (CHỈ chạy khi B4 đạt)

**Không ghi đè thư mục `data/silver/` đã có.** Cell này đổi tên các thư mục tạm đã kiểm tra thành ba dataset Silver chính thức. Chỉ sử dụng Silver khi có `reports/silver/after_review/silver_commit_manifest.json` xác nhận trạng thái `COMPLETE`.

Nếu đã có Silver từ notebook cũ, hãy **sao lưu/đổi tên thư mục cũ** trước; không để notebook tự xóa dữ liệu.

In [20]:
def b_commit_silver():
    if set(B_BUILT) != set(DATASETS):
        raise RuntimeError('Chưa có kết quả hợp lệ của đủ cả ba dataset')
    manifest_path = B_AUDIT_ROOT/'silver_commit_manifest.json'
    if manifest_path.exists():
        raise FileExistsError(f'Đã có manifest Silver: {manifest_path}; không chốt lần nữa')
    for name in DATASETS:
        src, dst = B_STAGE_ROOT/name, SILVER/name
        if not src.is_dir() or dst.exists():
            raise RuntimeError(f'{name}: source Silver tạm không tồn tại hoặc đích đã tồn tại: {src} → {dst}')
        for path, digest in B_REVIEWS[name]['source_files'].items():
            if file_sha256(path) != digest:
                raise RuntimeError(f'Phiếu review đổi sau khi dựng Silver tạm: {path}')
        n = spark.read.parquet(str(src)).count()
        if n != B_BUILT[name]['silver_rows']:
            raise RuntimeError(f'{name}: số dòng Silver tạm thay đổi: {n}')

    SILVER.mkdir(parents=True, exist_ok=True)
    committed = []
    try:
        for name in DATASETS:
            src, dst = B_STAGE_ROOT/name, SILVER/name
            src.rename(dst)  # cùng volume; không overwrite
            committed.append(name)
            print(f'ĐÃ CHỐT {name}: {B_BUILT[name]["silver_rows"]:,} dòng → {dst}')
        payload = {
            'status':'COMPLETE',
            'created_at_utc':datetime.now(timezone.utc).isoformat(),
            'datasets':{name:{'rows':B_BUILT[name]['silver_rows'],
                'split_counts':B_BUILT[name]['by_split'],
                'directory':str(SILVER/name)} for name in DATASETS},
            'notes': ['Bronze unchanged','VietNews word segmentation preserved',
                      'XSum: dropped 20 approved boilerplate source groups',
                      'Near-duplicate detection is not exhaustive: blocking can skip large blocks']}
        write_json(manifest_path,payload)
        print('SILVER ĐÃ HOÀN THÀNH. Manifest:',manifest_path)
    except Exception:
        print('COMMIT CHƯA HOÀN TẤT:', committed)
        print('Không dùng Silver nếu manifest chưa ghi status COMPLETE. Giữ nguyên để kiểm tra thủ công.')
        raise

b_commit_silver()


ĐÃ CHỐT cnn_dailymail: 308,449 dòng → D:\DuLieuCuaHuu\HK1_20262027\BigData\CK_BIG\data\silver\cnn_dailymail
ĐÃ CHỐT xsum: 226,413 dòng → D:\DuLieuCuaHuu\HK1_20262027\BigData\CK_BIG\data\silver\xsum
ĐÃ CHỐT vietnews: 143,785 dòng → D:\DuLieuCuaHuu\HK1_20262027\BigData\CK_BIG\data\silver\vietnews
SILVER ĐÃ HOÀN THÀNH. Manifest: D:\DuLieuCuaHuu\HK1_20262027\BigData\CK_BIG\reports\silver\after_review\silver_commit_manifest.json


## Kết thúc Notebook 02
- Bronze: **không chỉnh sửa**.
- Silver: `data/silver/cnn_dailymail/`, `data/silver/xsum/`, `data/silver/vietnews/`.
- Audit: `reports/silver/after_review/<dataset>/exclusions/`, báo cáo JSON/CSV và `silver_commit_manifest.json`.
- **Chỉ chuyển Notebook 03 — Gold khi manifest `COMPLETE`** và đã kiểm tra lại việc loại 20 nguồn XSum.

**Giới hạn phương pháp:** gần trùng chỉ được kiểm tra trên những cặp do thuật toán blocking đề xuất; 29 block CNN/DailyMail và 20 block XSum bị bỏ qua ở lần chạy này, vì vậy không thể khẳng định đã loại hết near-duplicate trên toàn corpus.